# Airbnb Data Ingestion to UC objects via UC volume 


In [0]:
# Unity Catalog target — files land in tanmay_projects.project_1.airbnb_raw
CATALOG = "tanmay_projects"
SCHEMA = "project_1"
VOLUME = "airbnb_raw"

VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# Cities exactly as named on https://insideairbnb.com/get-the-data/ (add more as needed)
CITIES = ["New York City"]

# Files to pull each run from each city's section on the page (summary "visualisations" set)
INCLUDE_FILES = {
    "listings.csv",        # listings summary
    "reviews.csv",          # reviews summary
    "neighbourhoods.csv",   # neighbourhood reference
    "neighbourhoods.geojson",
}

# Also download the detailed datasets (listings/calendar/reviews .csv.gz under /data/)?
# These are large (hundreds of MB per city) — enable only if you need them.
INCLUDE_DETAILED = False
DETAILED_FILES = {"listings.csv.gz", "calendar.csv.gz", "reviews.csv.gz"}

spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME} "
    "COMMENT 'Raw Inside Airbnb files, refreshed daily with overwrite'"
)
print(f"Volume ready: {VOLUME_PATH}")

# --- List all files currently available for New York on the Inside Airbnb data page ---
import re

import requests

GET_DATA_URL = "https://insideairbnb.com/get-the-data/"
html = requests.get(GET_DATA_URL, timeout=60).text

for city in CITIES:
    print(f"\n=== Files available for {city} on {GET_DATA_URL} ===")
    match = next(
        (
            m
            for m in re.finditer(r"<h3>([^<]+)</h3>", html, flags=re.IGNORECASE)
            if city.lower() in m.group(1).lower()
        ),
        None,
    )
    if match is None:
        print("NOT FOUND ON PAGE")
        continue
    section_end = min(
        [
            x
            for x in (
                html.find("<h3", match.end()),
                html.find('class="showArchivedData"', match.end()),
            )
            if x != -1
        ]
        or [len(html)]
    )
    for url in re.findall(r'href="(https://data\.insideairbnb\.com/[^"]+)"', html[match.end() : section_end]):
        print(url)

Volume ready: /Volumes/tanmay_projects/project_1/airbnb_raw

=== Files available for New York City on https://insideairbnb.com/get-the-data/ ===
https://data.insideairbnb.com/united-states/ny/new-york-city/2026-09-14/data/listings.csv.gz
https://data.insideairbnb.com/united-states/ny/new-york-city/2026-09-14/data/calendar.csv.gz
https://data.insideairbnb.com/united-states/ny/new-york-city/2026-09-14/data/reviews.csv.gz
https://data.insideairbnb.com/united-states/ny/new-york-city/2026-09-14/visualisations/listings.csv
https://data.insideairbnb.com/united-states/ny/new-york-city/2026-09-14/visualisations/reviews.csv
https://data.insideairbnb.com/united-states/ny/new-york-city/2026-09-14/visualisations/neighbourhoods.csv
https://data.insideairbnb.com/united-states/ny/new-york-city/2026-09-14/visualisations/neighbourhoods.geojson


In [0]:
import os
import re

import requests

GET_DATA_URL = "https://insideairbnb.com/get-the-data/"


def get_page_html(url: str = GET_DATA_URL) -> str:
    """Fetch the Inside Airbnb data page HTML."""
    resp = requests.get(url, timeout=60)
    resp.raise_for_status()
    return resp.text


def find_city_links(page_html: str, city: str) -> list:
    """Return the current data.insideairbnb.com download links from the city's section of the page.

    Each city has a section: <h3>City, State, Country</h3> followed by a table of
    file links. The section runs until the archived-data toggle / next city heading.
    """
    for m in re.finditer(r"<h3>([^<]+)</h3>", page_html, flags=re.IGNORECASE):
        if city.lower() not in m.group(1).lower():
            continue
        candidates = [
            x
            for x in (
                page_html.find("<h3", m.end()),
                page_html.find('class="showArchivedData"', m.end()),
            )
            if x != -1
        ]
        section_end = min(candidates) if candidates else len(page_html)
        return re.findall(r'href="(https://data\.insideairbnb\.com/[^"]+)"', page_html[m.end() : section_end])
    return []


def save_to_volume(url: str, dest: str) -> None:
    """Stream a remote file directly into the UC volume, overwriting any existing copy."""
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with requests.get(url, stream=True, timeout=600) as resp:
        resp.raise_for_status()
        with open(dest, "wb") as f:  # 'wb' overwrites any previous file
            for chunk in resp.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)


def download_city_files(city: str, page_html: str) -> list:
    """Download every file currently listed for a city into the volume, overwriting existing copies.

    Returns a list of (city, file, source_url, bytes_written) tuples.
    """
    links = find_city_links(page_html, city)
    if not links:
        return [(city, "NOT FOUND ON PAGE", "", 0)]

    slug = re.sub(r"[^a-z0-9]+", "-", city.lower()).strip("-")
    results = []
    for url in links:
        name = url.rstrip("/").split("/")[-1]
        dest = f"{VOLUME_PATH}/{slug}/{name}"
        save_to_volume(url, dest)
        results.append((city, name, url, os.path.getsize(dest)))
    return results


# --- Run the download: every configured file for every city, overwriting prior copies ---
html = get_page_html()

results = []
for city in CITIES:
    results.extend(download_city_files(city, html))

if not results:
    raise ValueError(f"No matching files found for cities {CITIES} — check the city names on {GET_DATA_URL}")

display(spark.createDataFrame(results, ["city", "file", "source_url", "bytes_written"]))

city,file,source_url,bytes_written
New York City,listings.csv.gz,https://data.insideairbnb.com/united-states/ny/new-york-city/2026-09-14/data/listings.csv.gz,16005851
New York City,calendar.csv.gz,https://data.insideairbnb.com/united-states/ny/new-york-city/2026-09-14/data/calendar.csv.gz,25603399
New York City,reviews.csv.gz,https://data.insideairbnb.com/united-states/ny/new-york-city/2026-09-14/data/reviews.csv.gz,120318684
New York City,listings.csv,https://data.insideairbnb.com/united-states/ny/new-york-city/2026-09-14/visualisations/listings.csv,5372531
New York City,reviews.csv,https://data.insideairbnb.com/united-states/ny/new-york-city/2026-09-14/visualisations/reviews.csv,22895274
New York City,neighbourhoods.csv,https://data.insideairbnb.com/united-states/ny/new-york-city/2026-09-14/visualisations/neighbourhoods.csv,4962
New York City,neighbourhoods.geojson,https://data.insideairbnb.com/united-states/ny/new-york-city/2026-09-14/visualisations/neighbourhoods.geojson,634167


In [0]:
import os


# --- Load raw files from the UC volume into Delta tables in tanmay_projects.project_1 ---
# Reads every CSV the ingestion step wrote under VOLUME_PATH/<city-slug>/ and loads it
# into a table named <city-slug>_<file-stem>, overwriting the table on every run.

def list_raw_files(volume_path: str = VOLUME_PATH) -> list:
    """Return absolute paths of all data files currently in the volume."""
    files = []
    for root, _, names in os.walk(volume_path):
        for name in names:
            files.append(os.path.join(root, name))
    return files


def load_file_to_table(src: str, catalog: str = CATALOG, schema: str = SCHEMA) -> str:
    """Load one raw file from the volume into a Delta table (overwrite mode).

    Table naming: <city-slug>_<file-stem>, e.g. new-york-city_listings -> tanmay_projects.project_1.new-york-city_listings
    """
    rel = os.path.relpath(src, VOLUME_PATH)  # e.g. "new-york-city/listings.csv"
    city_slug, filename = rel.split(os.sep, 1)
    stem = os.path.splitext(filename)[0].replace(".", "_")
    if filename.endswith(".geojson"):  # avoid colliding with the neighbourhoods.csv table
        stem += "_geojson"
    table_name = f"{city_slug}_{stem}".replace("-", "_")
    full_table = f"{catalog}.{schema}.{table_name}"

    if filename.endswith(".csv") or filename.endswith(".csv.gz"):
        df = (
            spark.read.format("csv")
            .option("header", True)
            .option("inferSchema", True)
            .option("multiLine", True)
            .load(src)
        )
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table)
    elif filename.endswith(".geojson"):
        df = spark.read.format("json").option("multiLine", True).load(src)
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table)
    else:
        print(f"Skipping unsupported file type: {rel}")
        return None
    return full_table


# --- Load everything currently in the volume into tables ---
loaded = []
for src in list_raw_files():
    table = load_file_to_table(src)
    if table:
        rows = spark.table(table).count()
        loaded.append((src, table, rows))

if not loaded:
    raise ValueError(f"No files found under {VOLUME_PATH} — run the ingestion cell first.")

display(spark.createDataFrame(loaded, ["source_file", "target_table", "row_count"]))


source_file,target_table,row_count
/Volumes/tanmay_projects/project_1/airbnb_raw/new-york-city/neighbourhoods.geojson,tanmay_projects.project_1.new_york_city_neighbourhoods_geojson,1
/Volumes/tanmay_projects/project_1/airbnb_raw/new-york-city/neighbourhoods.csv,tanmay_projects.project_1.new_york_city_neighbourhoods,230
/Volumes/tanmay_projects/project_1/airbnb_raw/new-york-city/listings.csv,tanmay_projects.project_1.new_york_city_listings,29918
/Volumes/tanmay_projects/project_1/airbnb_raw/new-york-city/reviews.csv,tanmay_projects.project_1.new_york_city_reviews,999099
/Volumes/tanmay_projects/project_1/airbnb_raw/new-york-city/listings.csv.gz,tanmay_projects.project_1.new_york_city_listings_csv,30522
/Volumes/tanmay_projects/project_1/airbnb_raw/new-york-city/calendar.csv.gz,tanmay_projects.project_1.new_york_city_calendar_csv,10920077
/Volumes/tanmay_projects/project_1/airbnb_raw/new-york-city/reviews.csv.gz,tanmay_projects.project_1.new_york_city_reviews_csv,999077
